# Kazakh ASR audit in Colab

This notebook is a safe, step-by-step workflow for running the ASR audit project in Google Colab without stressing your local laptop.

Checklist:
- install dependencies
- set Hugging Face cache in /content
- run a tiny smoke test
- run real data preparation
- transcribe with Whisper
- evaluate results
- save outputs to Google Drive if needed


In [ ]:
# 1) Mount Drive (optional but recommended)
from google.colab import drive

drive.mount('/content/drive')
print('Drive mounted.')


In [1]:
# 2) Get the project into the runtime
#
# Repo is private, so an unauthenticated clone fails. Paste a GitHub token
# when prompted (github.com/settings/tokens -> Generate new token (classic)
# -> scope "repo"). getpass hides the input and it is never written to disk
# or to this notebook file.

import os
from getpass import getpass

REPO_URL = 'https://github.com/assemqb/audit.git'
project_dir = '/content/kk-asr-audit'

if not os.path.exists(project_dir):
    token = getpass('GitHub token (repo scope): ')
    auth_url = REPO_URL.replace('https://', f'https://{token}@')
    !git clone "$auth_url" "$project_dir"
    del token, auth_url  # don't keep the token sitting around in memory

os.chdir(project_dir)
print('Project root:', project_dir)


GitHub token (repo scope): ··········
Cloning into '/content/kk-asr-audit'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 44 (delta 14), reused 36 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (44/44), 41.76 KiB | 13.92 MiB/s, done.
Resolving deltas: 100% (14/14), done.
Project root: /content/kk-asr-audit


In [2]:
# 3) Install dependencies in a safe, reproducible order
!pip install -U pip
!pip install -r requirements.txt

# torch/torchcodec are only used to decode FLEURS audio in prepare_data.py
# (a CPU-side task, nothing to do with Whisper's own GPU speed, which comes
# from ctranslate2). Installing from the CPU wheel index avoids a GPU build
# that expects a matching CUDA nvrtc runtime and fails with
# "libnvrtc.so.13: cannot open shared object file" if the versions don't line up.
!pip install --index-url https://download.pytorch.org/whl/cpu torch torchcodec

import os

# Same path prepare_data.py computes on its own (repo_root/.hf_cache) so every
# step shares one cache instead of downloading models/data twice.
hf_cache = os.path.join(project_dir, '.hf_cache')
os.environ['HF_HOME'] = hf_cache
os.environ['HUGGINGFACE_HUB_CACHE'] = os.path.join(hf_cache, 'hub')
os.environ['HF_DATASETS_CACHE'] = os.path.join(hf_cache, 'datasets')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(hf_cache, 'transformers')
os.environ['HF_HUB_DISABLE_XET'] = '1'

print('Dependencies installed and cache configured at', hf_cache)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 90.8 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 43.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 59.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 113.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 44.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 102.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [faster-whisper]
Looking in indexes: https://download.pytorch.org/whl/cpu
Dependencies installed and cache configured at /content/kk-asr-audit/.hf_cache


In [3]:
# 4) Safe smoke test: tiny data, tiny run
!python src/prepare_data.py --minutes 1 --split test --limit 1
!python src/transcribe.py --model small --lang kk --device auto --compute_type int8
!python src/evaluate.py --hyp results/hyp_small_kk.tsv
print('Smoke test finished.')


README.md: 100% 386k/386k [00:00<00:00, 152MB/s]
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/google/fleurs/resolve/70bb2e84b976b7e960aa89f1c648e09c59f894dd/parquet-data/kk_kz/test-00000-of-00001.parquet
Retrying in 1s [Retry 1/5].
Источник: fleurs (google/fleurs)
Записей: 1
Суммарная длительность: 0.25 мин
Эталоны: data/refs.tsv

Готово: results/hyp_small_kk.tsv
Аудио: 0.25 мин, время работы: 0.02 мин
RTF: 0.098
{
  "n_utt": 1,
  "audio_min": 0.25,
  "WER_raw": 0.7,
  "WER_norm": 0.6667,
  "WER_fold": 0.619,
  "WER_stem": 0.6667,
  "CER_norm": 0.1522,
  "delta_normalization": 0.0333,
  "delta_kk_graphemes": 0.0477,
  "delta_morphology": 0.0,
  "WER_norm_ci95": [
    NaN,
    NaN
  ],
  "WER_stem_ci95": [
    NaN,
    NaN
  ]
}

95% CI (бутстрап, 1000 прогонов, n=1 уттерансов):
  WER_norm: [nan, nan]
  WER_stem: [nan, nan]

Типы ошибок:
      8   57.1%  близкая форма (фонетика/опечатка)
      2   14.3%  пропуск слова
      2   14.3%  полная

In [4]:
# 5) Full run for the actual audit
# --device auto picks the Colab GPU automatically when the runtime has one
# (Runtime > Change runtime type > T4 GPU), otherwise falls back to CPU.
!python src/prepare_data.py --minutes 12 --split test
!python src/transcribe.py --model large-v3 --lang kk --device auto --compute_type int8_float16
!python src/transcribe.py --model small --lang kk --device auto --compute_type int8
!python src/transcribe.py --model large-v3 --lang none --device auto --compute_type int8_float16
!python src/evaluate.py --hyp results/hyp_large-v3_kk.tsv
print('Full audit finished.')


Источник: fleurs (google/fleurs)
Записей: 43
Суммарная длительность: 12.05 мин
Эталоны: data/refs.tsv
  20/43
  40/43

Готово: results/hyp_large-v3_kk.tsv
Аудио: 12.05 мин, время работы: 1.33 мин
RTF: 0.111
  20/43
  40/43

Готово: results/hyp_small_kk.tsv
Аудио: 12.05 мин, время работы: 0.53 мин
RTF: 0.044
  20/43
  40/43

Готово: results/hyp_large-v3_none.tsv
Аудио: 12.05 мин, время работы: 1.58 мин
RTF: 0.131
{
  "n_utt": 43,
  "audio_min": 12.05,
  "WER_raw": 0.3725,
  "WER_norm": 0.3055,
  "WER_fold": 0.2924,
  "WER_stem": 0.2833,
  "CER_norm": 0.0617,
  "delta_normalization": 0.067,
  "delta_kk_graphemes": 0.0131,
  "delta_morphology": 0.0222,
  "WER_norm_ci95": [
    0.2655,
    0.3475
  ],
  "WER_stem_ci95": [
    0.2438,
    0.3255
  ]
}

95% CI (бутстрап, 1000 прогонов, n=43 уттерансов):
  WER_norm: [0.2655, 0.3475]
  WER_stem: [0.2438, 0.3255]

Типы ошибок:
    140   59.8%  близкая форма (фонетика/опечатка)
     31   13.2%  полная замена слова
     18    7.7%  вставка слова


# Domain comparison: studio (FLEURS) vs crowdsourced (Common Voice)

FLEURS is a studio narrator reading in clean conditions — that's an upper
bound on quality, not a realistic one. No streamable *spontaneous* Kazakh
corpus was found (issai/Kazakh_Speech_Corpus_2 has TV/radio/podcast audio,
which would be closer, but ships as one unsegmented 80+ GB tar.gz with no
per-utterance split, not practical for a quick audit). Common Voice kk is
still read/prompted, not spontaneous, but it's crowdsourced — different
mics, devices, accents, background noise — so it tests something FLEURS
can't: does accuracy hold up outside a recording studio?

Same model/lang config as the FLEURS run above, so the two numbers are
directly comparable.


In [5]:
!python src/prepare_data.py --source common_voice --out data_cv --minutes 4 --split test
!python src/transcribe.py --model large-v3 --lang kk --device auto --compute_type int8_float16 --data data_cv --results results_cv
!python src/evaluate.py --hyp results_cv/hyp_large-v3_kk.tsv --data data_cv --results results_cv
print('Common Voice comparison run finished.')


README.md: 100% 504/504 [00:00<00:00, 1.67MB/s]
Источник: common_voice (Shirali/common_voice_11_0_kk)
Записей: 46
Суммарная длительность: 4.10 мин
Эталоны: data_cv/refs.tsv
  20/46
  40/46

Готово: results_cv/hyp_large-v3_kk.tsv
Аудио: 4.10 мин, время работы: 0.74 мин
RTF: 0.181
{
  "n_utt": 46,
  "audio_min": 4.1,
  "WER_raw": 0.6128,
  "WER_norm": 0.5382,
  "WER_fold": 0.5,
  "WER_stem": 0.4896,
  "CER_norm": 0.1393,
  "delta_normalization": 0.0746,
  "delta_kk_graphemes": 0.0382,
  "delta_morphology": 0.0486,
  "WER_norm_ci95": [
    0.4754,
    0.605
  ],
  "WER_stem_ci95": [
    0.4295,
    0.5587
  ]
}

95% CI (бутстрап, 1000 прогонов, n=46 уттерансов):
  WER_norm: [0.4754, 0.605]
  WER_stem: [0.4295, 0.5587]

Типы ошибок:
     93   60.0%  близкая форма (фонетика/опечатка)
     24   15.5%  полная замена слова
     10    6.5%  пропуск слова
     10    6.5%  спецбуквы (ә/ө/ұ/ү/і/ң/қ/ғ)
      9    5.8%  аффикс (морфология)
      5    3.2%  спецбуквы + аффикс
      2    1.3%  уход в 

In [6]:
# 6) Optional: save results to Drive
# Use if you want the outputs to survive session restarts.
# Run this in the SAME session as the runs above — /content is wiped on
# every runtime restart, so if the session already recycled, nothing here
# will exist and you'll need to rerun from cell 2.

drive_dir = '/content/drive/MyDrive/kk-asr-audit'
!mkdir -p "$drive_dir"
for d in ['data', 'results', 'data_cv', 'results_cv']:
    !test -e /content/kk-asr-audit/{d} && cp -r /content/kk-asr-audit/{d} "$drive_dir/" || echo "skip: {d} not found"
print('Results copied to Drive.')


Results copied to Drive.
